# Train Offline - UAV Forest Segmentation

## Internet TAT + GPU BAT

### Add Data:
1. Output cua notebook **Prepare Offline Assets** (tab Notebook Output)
2. Dataset chua forest data (seq1-seq9)

---
## Cell 1: CONFIG

In [ ]:
MODEL      = 'MIT-B5'       # MIT-B5 | MIT-B2 | MIT-B3
IMG_SIZE   = (768, 768)
BATCH_SIZE = 8
EPOCHS     = 100
LR         = 3e-4
FOCAL_LOSS = True
PATIENCE   = 20

RESUME = None

TRAIN_SEQS = ['seq1', 'seq2', 'seq3', 'seq4', 'seq5', 'seq7', 'seq8']
VAL_SEQS   = ['seq6']

_ZOO = {'MIT-B5': 'mit_b5', 'MIT-B2': 'mit_b2', 'MIT-B3': 'mit_b3'}
ENCODER = _ZOO[MODEL]
IMG_H, IMG_W = IMG_SIZE

print(f'MODEL: DeepLabV3+ ({ENCODER}) | {IMG_H}x{IMG_W} | BS={BATCH_SIZE}')

---
## Cell 2: Offline Setup

1. Tim `offline_assets/` trong `/kaggle/input/`
2. Cai smp tu `.whl`
3. Copy `dot_cache/` nguoc lai `~/.cache/` -> HuggingFace + Torch tim thay weights

In [ ]:
import os, sys, subprocess, shutil, glob
from pathlib import Path

print("[1/6] OFFLINE SETUP")

# === Tim offline_assets ===
ASSET_DIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "offline_assets" in dirs:
        ASSET_DIR = os.path.join(root, "offline_assets")
        break
    if "packages" in dirs and ("weights" in dirs or "dot_cache" in dirs):
        ASSET_DIR = root
        break

if not ASSET_DIR:
    raise RuntimeError("Khong tim thay offline_assets!")
print(f"  Found: {ASSET_DIR}")

# === Install smp ===
pkg_dir = os.path.join(ASSET_DIR, "packages")
if os.path.exists(pkg_dir) and glob.glob(os.path.join(pkg_dir, "*.whl")):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--no-index", "--find-links", pkg_dir,
        "segmentation-models-pytorch", "--quiet",
    ])
    print("  [OK] smp installed")

# === KHOI PHUC CACHE ===
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

home_cache = os.path.expanduser("~/.cache")
HF_HUB = os.path.join(home_cache, "huggingface", "hub")
TORCH_CKPT = os.path.join(home_cache, "torch", "hub", "checkpoints")
os.makedirs(HF_HUB, exist_ok=True)
os.makedirs(TORCH_CKPT, exist_ok=True)
copied = 0

# --- Cach 1: dot_cache/ (NB1 moi) ---
dot_cache = os.path.join(ASSET_DIR, "dot_cache")
if os.path.exists(dot_cache):
    print("  Using dot_cache/ (new format)")
    for root, dirs, files in os.walk(dot_cache):
        for f in files:
            src = os.path.join(root, f)
            rel = os.path.relpath(src, dot_cache)
            dst = os.path.join(home_cache, rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if not os.path.exists(dst):
                try: os.symlink(src, dst)
                except OSError: shutil.copy2(src, dst)
                copied += 1

# --- Cach 2: weights/hub/ (NB1 cu) ---
else:
    weights_hub = os.path.join(ASSET_DIR, "weights", "hub")
    if os.path.exists(weights_hub):
        print("  Using weights/hub/ (old format)")
        for item in os.listdir(weights_hub):
            src = os.path.join(weights_hub, item)
            if item.startswith("models--"):
                dst = os.path.join(HF_HUB, item)
                if not os.path.exists(dst):
                    try: os.symlink(src, dst)
                    except OSError: shutil.copytree(src, dst)
                    copied += 1
                    print(f"    -> {item}")
            elif item == "checkpoints":
                for ckpt in os.listdir(src):
                    s = os.path.join(src, ckpt)
                    d = os.path.join(TORCH_CKPT, ckpt)
                    if not os.path.exists(d):
                        try: os.symlink(s, d)
                        except OSError: shutil.copy2(s, d)
                        copied += 1
                        print(f"    -> checkpoints/{ckpt}")
    else:
        print("  [WARN] No weights found!")

print(f"  [OK] Restored {copied} items")

# Verify
if os.path.exists(HF_HUB):
    models = [d for d in os.listdir(HF_HUB) if d.startswith("models--")]
    print(f"  HF models: {models}")
if os.path.exists(TORCH_CKPT):
    files = os.listdir(TORCH_CKPT)
    if files: print(f"  Torch checkpoints: {files}")
print("  [OK] Done")


---
## Cell 3: Imports

In [ ]:
import re, time, json, math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

import segmentation_models_pytorch as smp
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('/kaggle/working/outputs')
CKPT_DIR = OUTPUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()} ({torch.cuda.get_device_properties(0).total_memory/1024**3:.0f} GB)')
else:
    print('[WARN] No GPU!')

---
## Cell 4: Google Drive Backup (optional)

In [ ]:
_gdrive = None
_folder_id = None
try:
    from kaggle_secrets import UserSecretsClient
    from google.oauth2.service_account import Credentials
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload
    s = UserSecretsClient()
    _gdrive = build('drive', 'v3', credentials=Credentials.from_service_account_info(
        json.loads(s.get_secret('GDRIVE_CREDENTIALS')),
        scopes=['https://www.googleapis.com/auth/drive.file']))
    _folder_id = s.get_secret('GDRIVE_FOLDER_ID')
    print('[OK] Google Drive enabled')
except:
    print('[SKIP] Google Drive')

def upload_to_drive(filepath):
    if not _gdrive: return
    try:
        f = Path(filepath)
        old = _gdrive.files().list(q=f"name='{f.name}' and '{_folder_id}' in parents and trashed=false", fields='files(id)').execute().get('files', [])
        media = MediaFileUpload(str(f), mimetype='application/octet-stream', resumable=True)
        if old: _gdrive.files().update(fileId=old[0]['id'], media_body=media).execute()
        else: _gdrive.files().create(body={'name': f.name, 'parents': [_folder_id]}, media_body=media).execute()
        print(f'    [DRIVE] {f.name}')
    except Exception as e:
        print(f'    [DRIVE ERR] {e}')

---
## Cell 5: Data

In [ ]:
print('[2/6] DATA')

NUM_CLASSES = 11
LABEL_COLORS = [
    (0,255,255), (0,127,0), (19,132,69), (0,53,65), (130,76,0),
    (152,251,152), (151,126,171), (250,150,0), (115,176,195),
    (123,123,123), (0,0,0),
]
CLASS_NAMES = [
    'Sky', 'Deciduous_trees', 'Coniferous_trees', 'Fallen_trees',
    'Dirt_ground', 'Ground_vegetation', 'Rocks', 'Building',
    'Fence', 'Car', 'Empty',
]

def rgb_to_mask(rgb):
    out = np.full(rgb.shape[:2], NUM_CLASSES - 1, dtype=np.int64)
    for i, c in enumerate(LABEL_COLORS):
        out[np.all(rgb == c, axis=-1)] = i
    return out

DATA_ROOT = Path('/kaggle/working/data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

found = []
for dirpath, dirnames, filenames in os.walk('/kaggle/input', followlinks=True):
    if os.path.basename(dirpath) == 'color':
        seq_dir = os.path.dirname(dirpath)
        seq_name = os.path.basename(seq_dir)
        if re.match(r'seq\d+', seq_name):
            link = DATA_ROOT / seq_name
            if not link.exists(): os.symlink(seq_dir, link)
            found.append(seq_name)
            print(f'  {seq_name} -> {seq_dir}')

seqs = sorted(set(found))
print(f'Found {len(seqs)} sequences')
if not seqs: raise RuntimeError('NO DATA!')

---
## Cell 6: Dataset + DataLoader

In [ ]:
class ForestDataset(Dataset):
    def __init__(self, root, sequences, transform=None):
        self.transform = transform
        self.samples = []
        root = Path(root)
        for seq in sequences:
            cd = root / seq / 'color'
            if not cd.exists(): continue
            for p in sorted(cd.glob('*.png')):
                lbl = root / seq / 'labels' / p.name
                if lbl.exists(): self.samples.append((p, lbl))
        print(f'  {len(self.samples)} samples from {sequences}')
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        img = np.array(Image.open(self.samples[i][0]).convert('RGB'))
        lbl = np.array(Image.open(self.samples[i][1]).convert('RGB'))
        mask = rgb_to_mask(lbl)
        if self.transform:
            r = self.transform(image=img, mask=mask)
            img, mask = r['image'], r['mask']
        return img.float(), mask.long()

train_seqs = [s for s in TRAIN_SEQS if s in seqs]
val_seqs = [s for s in VAL_SEQS if s in seqs]
if not train_seqs:
    train_seqs = seqs[:-1] if len(seqs) > 1 else seqs
    val_seqs = seqs[-1:] if len(seqs) > 1 else seqs
print(f'Train: {train_seqs} | Val: {val_seqs}')

train_tf = A.Compose([
    A.Resize(IMG_H, IMG_W), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.1),
    A.Rotate(limit=15, p=0.3, border_mode=0),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.3), A.HueSaturationValue(10, 20, 20, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.1),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2(),
])
val_tf = A.Compose([
    A.Resize(IMG_H, IMG_W),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2(),
])

train_ds = ForestDataset(DATA_ROOT, train_seqs, train_tf)
val_ds = ForestDataset(DATA_ROOT, val_seqs, val_tf)
if len(train_ds) == 0: raise RuntimeError('Train EMPTY!')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print(f'Train: {len(train_ds)} ({len(train_loader)} batches) | Val: {len(val_ds)}')

---
## Cell 7: Class Weights

In [ ]:
print('[3/6] CLASS WEIGHTS')
n_sample = min(200, len(train_ds))
counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for idx in np.random.choice(len(train_ds), n_sample, replace=False):
    _, mask = train_ds[idx]
    for c in range(NUM_CLASSES): counts[c] += (mask == c).sum().item()
freq = counts / counts.sum()
for i, name in enumerate(CLASS_NAMES): print(f'  {name:20s} {freq[i]*100:6.2f}%')
w = np.ones(NUM_CLASSES, dtype=np.float32)
nz = freq > 0
w[nz] = 1.0 / (freq[nz] * NUM_CLASSES)
w = np.clip(w, 0.5, 10.0)
w = w / w.sum() * NUM_CLASSES
CLASS_WEIGHTS = torch.tensor(w, device=DEVICE)
print(f'Weights: {w.round(2).tolist()}')

---
## Cell 8: Build Model

In [ ]:
print('[4/6] BUILD MODEL')
model = smp.DeepLabV3Plus(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES, activation=None)
print(f'  DeepLabV3+ ({ENCODER}): {sum(p.numel() for p in model.parameters()):,} params')
model = model.to(DEVICE)
print(f'  [OK] on {DEVICE}')

---
## Cell 9: Loss + Optimizer

In [ ]:
print('[5/6] LOSS + OPTIMIZER')

class FocalDiceLoss(nn.Module):
    def __init__(self, nc, cw=None, gamma=2.0, use_focal=True):
        super().__init__()
        self.nc, self.use_focal, self.gamma = nc, use_focal, gamma
        self.ce = nn.CrossEntropyLoss(weight=cw, reduction='none' if use_focal else 'mean')
    def forward(self, logits, targets):
        if self.use_focal:
            ce = self.ce(logits, targets)
            ce_term = ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()
        else: ce_term = self.ce(logits, targets)
        probs = F.softmax(logits, dim=1)
        tgt = F.one_hot(targets, self.nc).permute(0,3,1,2).float()
        inter = (probs * tgt).sum(dim=(0,2,3))
        card = probs.sum(dim=(0,2,3)) + tgt.sum(dim=(0,2,3))
        dice = 1.0 - ((2*inter + 1e-6) / (card + 1e-6)).mean()
        return ce_term + 0.5 * dice

criterion = FocalDiceLoss(NUM_CLASSES, CLASS_WEIGHTS, use_focal=FOCAL_LOSS)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
WARMUP = 5
scheduler = LambdaLR(optimizer, lambda e: e/WARMUP if e < WARMUP else 0.5*(1+math.cos(math.pi*(e-WARMUP)/max(1,EPOCHS-WARMUP))))
scaler = GradScaler('cuda', enabled=True)

start_epoch = 0
if RESUME and os.path.exists(RESUME):
    ckpt = torch.load(RESUME, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if 'scheduler_state_dict' in ckpt: scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = ckpt.get('epoch', 0)
    print(f'  [RESUME] epoch {start_epoch}')
print('  [OK]')

---
## Cell 10: Metrics

In [ ]:
class SegMetrics:
    def __init__(self, n):
        self.n = n; self.cm = np.zeros((n,n), dtype=np.int64)
    def reset(self): self.cm[:] = 0
    @torch.no_grad()
    def update(self, preds, targets):
        p, t = preds.cpu().numpy().flatten(), targets.cpu().numpy().flatten()
        v = (t >= 0) & (t < self.n)
        self.cm += np.bincount(t[v]*self.n + p[v], minlength=self.n**2).reshape(self.n, self.n)
    def compute(self):
        inter = np.diag(self.cm)
        union = self.cm.sum(1) + self.cm.sum(0) - inter
        iou = np.where(union > 0, inter/union, 0.0)
        valid = union > 0
        return {'miou': float(iou[valid].mean()) if valid.any() else 0.0,
                'pixel_acc': float(inter.sum()/self.cm.sum()) if self.cm.sum()>0 else 0.0,
                'iou': iou}

---
## Cell 11: TRAINING

In [ ]:
print('[6/6] TRAINING')
print('=' * 60)

best_miou = 0.0
no_improve = 0
saved_ckpts = []
metrics = SegMetrics(NUM_CLASSES)
history = {'train_loss': [], 'val_loss': [], 'miou': [], 'pixel_acc': [], 'lr': []}

for epoch in range(start_epoch + 1, EPOCHS + 1):
    t0 = time.time()

    model.train()
    tl, nt = 0.0, 0
    for imgs, masks in tqdm(train_loader, desc=f'E{epoch} train', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        with autocast('cuda'): loss = criterion(model(imgs), masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        tl += loss.item(); nt += 1
    train_loss = tl / nt

    model.eval()
    metrics.reset()
    vl, nv = 0.0, 0
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f'E{epoch} val', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            with autocast('cuda'):
                logits = model(imgs)
                vl += criterion(logits, masks).item(); nv += 1
            metrics.update(logits.argmax(1), masks)
    val_loss = vl / nv
    r = metrics.compute()
    miou, pa = r['miou'], r['pixel_acc']

    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    history['miou'].append(miou); history['pixel_acc'].append(pa); history['lr'].append(lr)

    print(f'E{epoch:3d}/{EPOCHS} | loss {train_loss:.4f}/{val_loss:.4f} | mIoU {miou:.4f} | pxAcc {pa:.4f} | lr {lr:.2e} | {time.time()-t0:.0f}s')

    if miou > best_miou:
        best_miou = miou; no_improve = 0
        path = CKPT_DIR / f'{MODEL}_e{epoch:03d}_miou{miou:.4f}.pth'
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'scaler_state_dict': scaler.state_dict(), 'miou': miou,
                    'config': {'model': MODEL, 'encoder': ENCODER, 'img_size': IMG_SIZE}}, path)
        saved_ckpts.append((miou, path))
        saved_ckpts.sort(key=lambda x: x[0])
        while len(saved_ckpts) > 3:
            _, old = saved_ckpts.pop(0)
            if old.exists(): old.unlink()
        upload_to_drive(str(path))
        print(f'  >>> BEST mIoU: {miou:.4f}')
        for i, name in enumerate(CLASS_NAMES):
            if r['iou'][i] > 0: print(f'      {name:20s}: {r["iou"][i]:.4f}')
    else:
        no_improve += 1
    if no_improve >= PATIENCE:
        print(f'Early stopping (patience={PATIENCE})'); break

print(f'\nDone! Best mIoU: {best_miou:.4f}')

---
## Cell 12: Results

In [ ]:
with open(OUTPUT_DIR / f'{MODEL}_history.json', 'w') as f: json.dump(history, f, indent=2)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 4, figsize=(22, 5))
ax[0].plot(history['train_loss'], label='Train'); ax[0].plot(history['val_loss'], label='Val')
ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(True, alpha=0.3)
ax[1].plot(history['miou'], 'g'); ax[1].axhline(best_miou, color='r', ls='--', alpha=0.5)
ax[1].set_title(f'mIoU (best={best_miou:.4f})'); ax[1].grid(True, alpha=0.3)
ax[2].plot(history['pixel_acc'], 'b'); ax[2].set_title('Pixel Acc'); ax[2].grid(True, alpha=0.3)
ax[3].plot(history['lr'], 'orange'); ax[3].set_title('LR'); ax[3].grid(True, alpha=0.3)
fig.suptitle(f'DeepLabV3+ ({ENCODER}) @ {IMG_H}x{IMG_W}', fontweight='bold')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / f'{MODEL}_curves.png', dpi=150); plt.show()
if saved_ckpts: print(f'Best: {saved_ckpts[-1][1].name}')